In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_scatter import scatter
from torch_geometric.utils import softmax as pyg_softmax
import timeit

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# dummy dataset 

In [2]:
def create_dummy_compound_graph(num_nodes=25, num_edges=50):
    x = torch.randn(num_nodes, 44)
    edge_index = torch.randint(0, num_nodes, (2, num_edges))
    edge_attr = torch.randn(num_edges, 10)
    lap_pos_enc = torch.randn(num_nodes, 8)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lap_pos_enc=lap_pos_enc)


def create_dummy_protein_graph(num_nodes=150, num_edges=300):
    x = torch.randn(num_nodes, 41)
    edge_index = torch.randint(0, num_nodes, (2, num_edges))
    edge_attr = torch.randn(num_edges, 5)
    lap_pos_enc = torch.randn(num_nodes, 8)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, lap_pos_enc=lap_pos_enc)


def create_dummy_protein_embedding(num_residues=150):
    return np.random.randn(num_residues, 320).astype(np.float32)


def create_dummy_dataset(num_samples=20):
    dataset = []
    for _ in range(num_samples):
        compound_nodes = np.random.randint(15, 40)
        compound_edges = compound_nodes * 2
        protein_nodes = np.random.randint(100, 200)
        protein_edges = protein_nodes * 2
        compound_graph = create_dummy_compound_graph(compound_nodes, compound_edges)
        protein_graph = create_dummy_protein_graph(protein_nodes, protein_edges)
        protein_embedding = create_dummy_protein_embedding(protein_nodes)
        label = np.random.uniform(4.0, 10.0)
        dataset.append({
            'compound_graph': compound_graph,
            'protein_graph': protein_graph,
            'protein_embedding': protein_embedding,
            'label': label
        })
    return dataset


class DTIDatasetPyG(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        return item['compound_graph'], item['protein_graph'], item['protein_embedding'], item['label']


def custom_collate_pyg(batch):  
    compound_graphs = [item[0] for item in batch]
    protein_graphs = [item[1] for item in batch]
    protein_embeddings = [item[2] for item in batch]
    labels = [item[3] for item in batch]

    compound_batch = Batch.from_data_list(compound_graphs)
    protein_batch = Batch.from_data_list(protein_graphs)


    protein_lens = [g.num_nodes for g in protein_graphs]
    max_protein_len = max(protein_lens)

    # max_protein_len = max(emb.shape[0] for emb in protein_embeddings)
    padded_embeddings = []
    for emb in protein_embeddings:
        if emb.shape[0] < max_protein_len:
            pad_size = max_protein_len - emb.shape[0]
            padded = np.pad(emb, ((0, pad_size), (0, 0)), mode='constant', constant_values=0)
            padded_embeddings.append(padded)
        else:
            padded_embeddings.append(emb)

    protein_embedding_batch = torch.FloatTensor(np.array(padded_embeddings))
    label_batch = torch.FloatTensor(labels)
    return compound_batch, protein_batch, protein_embedding_batch, label_batch


def demonstrate_shapes():
    dummy_data = create_dummy_dataset(num_samples=20)
    print(f"Created {len(dummy_data)} samples. Sample 0 embedding shape: {dummy_data[0]['protein_embedding'].shape}")

    dataset = DTIDatasetPyG(dummy_data)
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_pyg, drop_last=True)
    for batch_idx, data in enumerate(dataloader):
        compound_graph, protein_graph, protein_embedding, label = data
        print(f"Batch {batch_idx + 1}: compound_nodes={compound_graph.x.shape[0]}, protein_nodes={protein_graph.x.shape[0]}, embedding_shape={protein_embedding.shape}, labels={label.tolist()}")
        print(compound_graph.edge_index)
        break



In [3]:
# demonstrate_shapes()

# graph_transformer_edge_layer

In [ ]:
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads, use_bias):
        super().__init__()
        
        self.out_dim = out_dim
        self.num_heads = num_heads
        
        if use_bias:
            self.Q = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.K = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.V = nn.Linear(in_dim, out_dim * num_heads, bias=True)
            self.proj_e = nn.Linear(in_dim, out_dim * num_heads, bias=True)
        else:
            self.Q = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.K = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.V = nn.Linear(in_dim, out_dim * num_heads, bias=False)
            self.proj_e = nn.Linear(in_dim, out_dim * num_heads, bias=False)
    
    def forward(self, h, edge_index, e):
        num_nodes = h.size(0)
        num_edges = edge_index.size(1)
        
        # Project to Q, K, V
        Q_h = self.Q(h).view(num_nodes, self.num_heads, self.out_dim)
        K_h = self.K(h).view(num_nodes, self.num_heads, self.out_dim)
        V_h = self.V(h).view(num_nodes, self.num_heads, self.out_dim)
        proj_e = self.proj_e(e).view(num_edges, self.num_heads, self.out_dim)
        
        # Get source and target node indices
        src, dst = edge_index[0], edge_index[1]
        
        # Gather K, Q, V for each edge
        K_j = K_h[src]  # [num_edges, num_heads, out_dim]
        Q_i = Q_h[dst]  # [num_edges, num_heads, out_dim]
        V_j = V_h[src]  # [num_edges, num_heads, out_dim]
        
        # Compute attention scores: Q·K / sqrt(d_k)
        score = (Q_i * K_j) / (self.out_dim ** 0.5)  # [num_edges, num_heads, out_dim]
        
        # Modulate by edge features (element-wise multiplication)
        score = score * proj_e  # [num_edges, num_heads, out_dim]
        
        # Store edge output (before softmax)
        e_out = score
        
        # Sum across feature dimension for softmax
        score_scalar = score.sum(dim=-1)  # [num_edges, num_heads]
        
        # Clamp for numerical stability
        score_scalar = score_scalar.clamp(-5, 5)
        
        # Apply softmax per target node for each head separately
        attn_weights = torch.zeros_like(score_scalar)
        for head in range(self.num_heads):
            attn_weights[:, head] = pyg_softmax(score_scalar[:, head], dst, num_nodes=num_nodes)
        
        # Expand for multiplication with V
        attn = attn_weights.unsqueeze(-1)  # [num_edges, num_heads, 1]
        
        # Weight the values
        msg = V_j * attn  # [num_edges, num_heads, out_dim]
        
        # Aggregate messages to each node
        h_out = scatter(msg, dst, dim=0, dim_size=num_nodes, reduce='sum')
        
        # Compute normalization factor (sum of attention weights)
        z = scatter(attn, dst, dim=0, dim_size=num_nodes, reduce='sum')
        
        # Normalize
        h_out = h_out / (z + 1e-6)
        
        return h_out, e_out

In [5]:
class GraphTransformerLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num_heads, dropout, layer_norm=True, batch_norm=False, residual=True, use_bias=False):
        super().__init__()
        self.in_channels = in_dim
        self.out_channels = out_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.residual = residual
        self.layer_norm = layer_norm
        self.batch_norm = batch_norm
        
        self.attention = MultiHeadAttentionLayer(in_dim, out_dim // num_heads, num_heads, use_bias)
        
        self.O_h = nn.Linear(out_dim, out_dim)
        self.O_e = nn.Linear(out_dim, out_dim)
        
        if self.layer_norm:
            self.layer_norm1_h = nn.LayerNorm(out_dim)
            self.layer_norm1_e = nn.LayerNorm(out_dim)
        
        if self.batch_norm:
            self.batch_norm1_h = nn.BatchNorm1d(out_dim)
            self.batch_norm1_e = nn.BatchNorm1d(out_dim)
        
        self.FFN_h_layer1 = nn.Linear(out_dim, out_dim * 2)
        self.FFN_h_layer2 = nn.Linear(out_dim * 2, out_dim)
        
        self.FFN_e_layer1 = nn.Linear(out_dim, out_dim * 2)
        self.FFN_e_layer2 = nn.Linear(out_dim * 2, out_dim)
        
        if self.layer_norm:
            self.layer_norm2_h = nn.LayerNorm(out_dim)
            self.layer_norm2_e = nn.LayerNorm(out_dim)
        
        if self.batch_norm:
            self.batch_norm2_h = nn.BatchNorm1d(out_dim)
            self.batch_norm2_e = nn.BatchNorm1d(out_dim)
    
    def forward(self, h, edge_index, e):
        h_in1 = h
        e_in1 = e
        
        # Multi-head attention
        h_attn_out, e_attn_out = self.attention(h, edge_index, e)
        
        h = h_attn_out.view(-1, self.out_channels)
        e = e_attn_out.view(-1, self.out_channels)
        
        h = F.dropout(h, self.dropout, training=self.training)
        e = F.dropout(e, self.dropout, training=self.training)
        
        h = self.O_h(h)
        e = self.O_e(e)
        
        if self.residual:
            h = h_in1 + h
            e = e_in1 + e
        
        if self.layer_norm:
            h = self.layer_norm1_h(h)
            e = self.layer_norm1_e(e)
        
        if self.batch_norm:
            h = self.batch_norm1_h(h)
            e = self.batch_norm1_e(e)
        
        h_in2 = h
        e_in2 = e
        
        # FFN for h
        h = self.FFN_h_layer1(h)
        h = F.relu(h)
        h = F.dropout(h, self.dropout, training=self.training)
        h = self.FFN_h_layer2(h)
        
        # FFN for e
        e = self.FFN_e_layer1(e)
        e = F.relu(e)
        e = F.dropout(e, self.dropout, training=self.training)
        e = self.FFN_e_layer2(e)
        
        if self.residual:
            h = h_in2 + h
            e = e_in2 + e
        
        if self.layer_norm:
            h = self.layer_norm2_h(h)
            e = self.layer_norm2_e(e)
        
        if self.batch_norm:
            h = self.batch_norm2_h(h)
            e = self.batch_norm2_e(e)
        
        return h, e

In [6]:
class GraphTransformerCompound(nn.Module):
    def __init__(self, device, n_layers, node_dim, edge_dim, hidden_dim, out_dim, n_heads, in_feat_dropout, dropout, pos_enc_dim):
        super().__init__()
        self.device = device
        self.layer_norm = True
        self.batch_norm = False
        self.residual = True
        self.linear_h = nn.Linear(node_dim, hidden_dim)
        self.linear_e = nn.Linear(edge_dim, hidden_dim)
        self.in_feat_dropout = nn.Dropout(in_feat_dropout)
        self.embedding_lap_pos_enc = nn.Linear(pos_enc_dim, hidden_dim)
        
        self.layers = nn.ModuleList([
            GraphTransformerLayer(hidden_dim, hidden_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
            for _ in range(n_layers - 1)
        ])
        self.layers.append(
            GraphTransformerLayer(hidden_dim, out_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
        )
    
    def forward(self, data):
        data = data.to(self.device)
        # print("here ")
        h = data.x.float().to(self.device)
        # print(h.shape)
        h_lap_pos_enc = data.lap_pos_enc.to(self.device)
        e = data.edge_attr.float().to(self.device)
        edge_index = data.edge_index.to(self.device)
        
        h = self.linear_h(h)
        h_lap_pos_enc = self.embedding_lap_pos_enc(h_lap_pos_enc.float())
        h = h + h_lap_pos_enc
        h = self.in_feat_dropout(h)
        
        e = self.linear_e(e)
        
        for conv in self.layers:
            h, e = conv(h, edge_index, e)
        
        return h
    

In [7]:
class GraphTransformerProtein(nn.Module):
    def __init__(self, device, n_layers, node_dim, edge_dim, hidden_dim, out_dim, n_heads, in_feat_dropout, dropout, pos_enc_dim):
        super().__init__()
        self.device = device
        self.layer_norm = True
        self.batch_norm = False
        self.residual = True
        self.linear_h = nn.Linear(node_dim, hidden_dim)
        self.linear_e = nn.Linear(edge_dim, hidden_dim)
        self.in_feat_dropout = nn.Dropout(in_feat_dropout)
        self.embedding_lap_pos_enc = nn.Linear(pos_enc_dim, hidden_dim)
        
        self.layers = nn.ModuleList([
            GraphTransformerLayer(hidden_dim, hidden_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
            for _ in range(n_layers - 1)
        ])
        self.layers.append(
            GraphTransformerLayer(hidden_dim, out_dim, n_heads, dropout, self.layer_norm, self.batch_norm, self.residual)
        )
    
    def forward(self, data):
        data = data.to(self.device)
        h = data.x.float().to(self.device)
        h_lap_pos_enc = data.lap_pos_enc.to(self.device)
        e = data.edge_attr.float().to(self.device)
        edge_index = data.edge_index.to(self.device)
        
        h = self.linear_h(h)
        h_lap_pos_enc = self.embedding_lap_pos_enc(h_lap_pos_enc.float())
        h = h + h_lap_pos_enc
        h = self.in_feat_dropout(h)
        
        e = self.linear_e(e)
        
        for conv in self.layers:
            h, e = conv(h, edge_index, e)
        
        return h

In [8]:
class SelfAttention(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout):
        super().__init__()
        
        self.hid_dim = hid_dim
        self.n_heads = n_heads
        assert hid_dim % n_heads == 0
        
        self.w_q = nn.Linear(hid_dim, hid_dim)
        self.w_k = nn.Linear(hid_dim, hid_dim)
        self.w_v = nn.Linear(hid_dim, hid_dim)
        self.fc = nn.Linear(hid_dim, hid_dim)
        self.do = nn.Dropout(dropout)
        self.scale = torch.sqrt(torch.FloatTensor([hid_dim // n_heads])).to(device)
    
    def forward(self, query, key, value, mask=None):
        bsz = query.shape[0]
        
        Q = self.w_q(query)
        K = self.w_k(key)
        V = self.w_v(value)
        
        Q = Q.view(bsz, -1, self.n_heads, self.hid_dim // self.n_heads).permute(0, 2, 1, 3)
        K = K.view(bsz, -1, self.n_heads, self.hid_dim // self.n_heads).permute(0, 2, 1, 3)
        V = V.view(bsz, -1, self.n_heads, self.hid_dim // self.n_heads).permute(0, 2, 1, 3)
        
        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale
        
        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)
        
        attention = self.do(F.softmax(energy, dim=-1))
        x = torch.matmul(attention, V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(bsz, -1, self.n_heads * (self.hid_dim // self.n_heads))
        x = self.fc(x)
        
        return x

In [9]:
def pyg_batch_split(bg, feats):  # Even named 'bg' like DGL!
    """
    Exact DGL logic, just swapping data accessors
    """
    # Only difference: PyG needs to compute batch_num_nodes
    batch = bg.batch
    batch_num_nodes = scatter(
        torch.ones_like(batch), batch, 
        dim=0, dim_size=bg.num_graphs, reduce='sum'
    )
    
    # Now follow DGL exactly
    max_num_nodes = int(batch_num_nodes.max())
    cum_nodes = torch.cat([batch.new_zeros(1), batch_num_nodes.cumsum(dim=0)])
    idx = torch.arange(feats.size(0), dtype=torch.long, device=feats.device)
    idx = (idx - cum_nodes[batch]) + (batch * max_num_nodes)
    size = [bg.num_graphs * max_num_nodes] + list(feats.size())[1:]
    out = feats.new_full(size, fill_value=0)
    out[idx] = feats
    out = out.view([bg.num_graphs, max_num_nodes] + list(feats.size())[1:])
    return out

In [ ]:
class MDGTDTInet(nn.Module):
    def __init__(self, compound_dim=128, protein_dim=128, gt_layers=10, gt_heads=8, out_dim=1):
        super(MDGTDTInet, self).__init__()
        self.compound_dim = compound_dim
        self.protein_dim = protein_dim
        self.n_layers = gt_layers
        self.n_heads = gt_heads
        
        self.crossAttention = SelfAttention(hid_dim=self.compound_dim, n_heads=1, dropout=0.2)
        
        self.compound_gt = GraphTransformerCompound(
            device, n_layers=gt_layers, node_dim=44, edge_dim=10, 
            hidden_dim=compound_dim, out_dim=compound_dim, n_heads=gt_heads, 
            in_feat_dropout=0.0, dropout=0.2, pos_enc_dim=8
        )
        
        self.protein_gt = GraphTransformerProtein(
            device, n_layers=gt_layers, node_dim=41, edge_dim=5, 
            hidden_dim=protein_dim, out_dim=protein_dim, n_heads=gt_heads, 
            in_feat_dropout=0.0, dropout=0.2, pos_enc_dim=8
        )
        
        self.protein_embedding_fc = nn.Linear(320, self.protein_dim)
        
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()
        self.joint_attn_prot = nn.Linear(compound_dim, compound_dim)
        self.joint_attn_comp = nn.Linear(compound_dim, compound_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(compound_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, out_dim)
        )
    
    def forward(self, compound_graph, protein_graph, protein_embedding):
        # Compound encoding
        compound_feat = self.compound_gt(compound_graph)
        compound_feat_x = pyg_batch_split(compound_graph, compound_feat)
        compound_feats = compound_feat_x
        
        # Protein encoding
        protein_feat = self.protein_gt(protein_graph)
        protein_feat_x = pyg_batch_split(protein_graph, protein_feat)
        protein_embedding = self.protein_embedding_fc(protein_embedding)
        protein_feats = self.crossAttention(protein_embedding, protein_feat_x, protein_feat_x)
        
        # Compound-protein interaction
        inter_comp_prot = self.sigmoid(
            torch.einsum('bij,bkj->bik', 
                        self.joint_attn_prot(self.relu(protein_feats)), 
                        self.joint_attn_comp(self.relu(compound_feats)))
        )
        inter_comp_prot_sum = torch.einsum('bij->b', inter_comp_prot)
        inter_comp_prot = torch.einsum('bij,b->bij', inter_comp_prot, 1/inter_comp_prot_sum)
        
        # Compound-protein joint embedding
        cp_embedding = self.tanh(torch.einsum('bij,bkj->bikj', protein_feats, compound_feats))
        cp_embedding = torch.einsum('bijk,bij->bk', cp_embedding, inter_comp_prot)
        
        x = self.classifier(cp_embedding)
        
        return x